# Streaming with LangSmith API

## Steps: 
1. Have Agent.py file ready in the studio folder
2. keep .env file with LangSmith API keys and GOOGle API keys ready
3. set langgraph.json file ready
4. To start the local development server,
   run the following command in your terminal in the /studio directory in this module: BAsh
   ' langgraph dev '

### check for output:
 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs

This in-memory server is designed for development and testing.
For production use, please use LangSmith Deployment.

6. Copy url = http://127.0.0.1:2024

#### Note: 
- check for more info : https://docs.langchain.com/langsmith/quick-start-studio#local-development-server
- check for LangSmith setup guide in the begining of the course.

7. ### Connecting to the Local Server Using the SDK
You use the LangGraph SDK to talk to your local agent: python
```
from langgraph_sdk import get_client

url = "http://127.0.0.1:2024"
client = get_client(url=url)
```
This client object lets you:

    - list assistants
    - create threads
    - run your agent
    - stream results

8. ### Create a thread for storing event checkpoints?
A thread is a conversation session.

It stores:

    - messages
    - memory
    - state
    - checkpoints

You create one like this: python
```
thread = await client.threads.create()
```
This returns: python
```
{"thread_id": "abc123"}
```
You will use this thread ID for the entire conversation.

9.  ### Running the Agent With Streaming
You run your agent like this: python
```
async for chunk in client.runs.stream(
        thread['thread_id'],   # which conversation
        "agent",               # which assistant
        input=msg_input,       # what the user said
        stream_mode="values",  # stream full state after each step
    ):
```
What happens?
Your agent runs step by step:

    - LLM node
    - Tool node
    - LLM node
    - Final answer
After each step, LangGraph sends you a chunk containing the updated state.

In [5]:
from langgraph_sdk import get_client

url = "http://127.0.0.1:2024"
client  = get_client(url=url)

In [6]:
# Search all hosted graphs
assistants = await client.assistants.search()
# list the agents
assistants

[{'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
  'graph_id': 'agent',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'agent',
  'created_at': '2026-04-27T16:29:37.327724+00:00',
  'updated_at': '2026-04-27T16:29:37.327724+00:00',
  'version': 1,
  'description': None}]

In [7]:
# create a thread
thread = await client.threads.create()
print(thread)

{'thread_id': '019dd0df-9c0b-7420-8069-430ab4a6b37f', 'created_at': '2026-04-27T21:36:48.651227+00:00', 'updated_at': '2026-04-27T21:36:48.651227+00:00', 'state_updated_at': '2026-04-27T21:36:48.651227+00:00', 'metadata': {}, 'status': 'idle', 'config': {}, 'values': None}


In [10]:
# stream events
from langchain_core.messages import HumanMessage
msg_input = [ HumanMessage(content="Multiply 3 and 10")]
async for event in client.runs.stream(
        thread["thread_id"],   # which conversation
        "agent",               # which assistant
        input={"messages": msg_input},       # what the user said
        stream_mode="values",  # stream full state after each step
    ):
    print(event)

StreamPart(event='metadata', data={'run_id': '019dd0e6-8b68-7b22-8ff4-7c1d88485850', 'attempt': 1}, id=None)
StreamPart(event='values', data={'messages': [{'content': 'Multiply 3 and 10', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': '4c09ff35-e153-4a7f-b131-a4701003e5b0'}]}, id=None)
StreamPart(event='values', data={'messages': [{'content': 'Multiply 3 and 10', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': '4c09ff35-e153-4a7f-b131-a4701003e5b0'}, {'content': [], 'additional_kwargs': {'function_call': {'name': 'multiply', 'arguments': '{"a": 3, "b": 10}'}, '__gemini_function_call_thought_signatures__': {'37bbae03-0a09-4c4d-b645-366f123649a2': 'EjQKMgEMOdbHcE03B+zU+cKpQU7rSLRRJY8jA4BbAu1nTa3FQhtJIXy9TXhfidbEYbM8Af5Z'}}, 'response_metadata': {'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, 'type': 'ai', 'name': None, 'id': 'lc_

### Note: 
- As you can see streamed obejects have event that is type and data  = state with field values
- Event is streamed as the state gets updated


#### Stream only messages with from langchain_core.messages import convert_to_messages

In [12]:
from langchain_core.messages import convert_to_messages
# create a new thread to avoid state mix up

thread = await client.threads.create()
input_message = HumanMessage(content="Multiply 20 and 3")
async for event in client.runs.stream(thread["thread_id"], assistant_id="agent", input={"messages": [input_message]}, stream_mode="values"):
    messages = event.data.get('messages',None)
    if messages:
        print(convert_to_messages(messages)[0])
    print('='*25)

content='Multiply 20 and 3' additional_kwargs={} response_metadata={} id='88e8af04-293e-444f-a1e3-04b4af5302d4'
content='Multiply 20 and 3' additional_kwargs={} response_metadata={} id='88e8af04-293e-444f-a1e3-04b4af5302d4'
content='Multiply 20 and 3' additional_kwargs={} response_metadata={} id='88e8af04-293e-444f-a1e3-04b4af5302d4'
content='Multiply 20 and 3' additional_kwargs={} response_metadata={} id='88e8af04-293e-444f-a1e3-04b4af5302d4'


# Streaming mode in LangSmith API

There are some new streaming mode that are only supported via the API.

For example, we can use *"messages"* mode to better handle the above case!
i.e,  stream_mode = 'messages'

This mode currently assumes that you have a messages key in your graph, which is a list of messages.

#### All events emitted using messages mode have two attributes:
- event: This is the name of the event
- data: This is data associated with the event

#### Note:
- metadata: metadata about the run
- messages/complete: fully formed message
- messages/partial: chat model tokens

In [15]:
# create a new thread to avoid state mix up

thread = await client.threads.create()
input_message = HumanMessage(content="add 100 and 50")
async for event in client.runs.stream(thread["thread_id"], assistant_id="agent", input={"messages": [input_message]}, stream_mode="messages"):
    print(event.event)

metadata
messages/metadata
messages/partial
messages/partial
messages/partial
messages/metadata
messages/complete
messages/metadata
messages/partial
messages/partial
messages/partial
messages/partial


In [23]:
thread = await client.threads.create()
input_message = HumanMessage(content="multiply 10 and 50")
events= []
async for event in client.runs.stream(thread["thread_id"], assistant_id="agent", input={"messages": [input_message]}, stream_mode="messages"):
    events.append(event)
    print(event)
    print("*"*30)

StreamPart(event='metadata', data={'run_id': '019dd1e5-4634-7463-b528-48897a5c327b', 'attempt': 1}, id=None)
******************************
StreamPart(event='messages/metadata', data={'lc_run--019dd1e5-5733-7870-b9ff-021f79c88df3': {'metadata': {'created_by': 'system', 'graph_id': 'agent', 'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca', 'run_attempt': 1, 'langgraph_version': '1.1.9', 'langgraph_api_version': '0.8.1', 'langgraph_plan': 'enterprise', 'langgraph_host': 'self-hosted', 'langgraph_api_url': 'http://127.0.0.1:2024', 'run_id': '019dd1e5-4634-7463-b528-48897a5c327b', 'thread_id': '019dd1e5-4631-75a2-8e94-7c882b576a3a', 'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'llm_assistant', 'langgraph_triggers': ['branch:to:llm_assistant'], 'langgraph_path': ['__pregel_pull', 'llm_assistant'], 'langgraph_checkpoint_ns': 'llm_assistant:4999f83a-e72d-8153-29ae-c34ff07bf9a6', 'checkpoint_ns': 'llm_assistant:4999f83a-e72d-8153-29ae-c34ff07bf9a6', 'l

#### Print information
1. MeataData Run ID
2. Tool Calls
3. Invalid tool Calls
4. Usage metadata - finish reason , model_name
6. Response metadata
7. function call - name

In [28]:
for event in events:
    if event.event == 'metadata':
       print(f"MetaData- Run-ID: {event.data['run_id']}")

MetaData- Run-ID: 019dd1e5-4634-7463-b528-48897a5c327b


In [35]:
for event in events:
    if event.event == 'messages/partial':
        content = event.data[0].get('content',"")
        tool_calls = event.data[0].get('tool_calls',[])
        invalid_tool_calls = event.data[0].get('invalid_tool_calls',[])
        response_metadata = event.data[0].get('response_metadata',{})
        usage_metadata = event.data[0].get('usage_metadata',{})
        if content:
            print("AI-content: ", content[0]['text'])
        if tool_calls:
            print("Tool Calls:")
            print(f"Tool_Call_Name: {tool_calls[-1]['name']}, Tool_Call_ID: {tool_calls[-1]['id']}, Arg: {tool_calls[-1]['args']}")
        if invalid_tool_calls :
            print("Invalid Tool Calls:")
            print(f"Tool_Call_Name: {tool_calls[-1]['name']}, Tool_Call_ID: {tool_calls[-1]['id']}, Arg: {tool_calls[-1]['args']}")
        if response_metadata:
            print("Response MetaData:")
            print(f"model_name: {response_metadata.get('model_name',"None")}, Finish Reason: {response_metadata.get('finish_reason',"None")}")
        if usage_metadata:
            print("Usage MetaData:")
            print(f"Input Tokens: {usage_metadata['input_tokens']}, Output Tokens: {usage_metadata['output_tokens']}, Total Tokens: {usage_metadata['total_tokens']}")

        print("-"*50)       
            
        
    

Tool Calls:
Tool_Call_Name: multiply, Tool_Call_ID: 8efbdc8e-71fe-4a6d-b6bd-7f415a2c18a3, Arg: {'b': 50, 'a': 10}
Response MetaData:
model_name: None, Finish Reason: None
Usage MetaData:
Input Tokens: 254, Output Tokens: 18, Total Tokens: 272
--------------------------------------------------
Tool Calls:
Tool_Call_Name: multiply, Tool_Call_ID: 8efbdc8e-71fe-4a6d-b6bd-7f415a2c18a3, Arg: {'b': 50, 'a': 10}
Response MetaData:
model_name: gemini-3.1-flash-lite-preview, Finish Reason: STOP
Usage MetaData:
Input Tokens: 254, Output Tokens: 18, Total Tokens: 272
--------------------------------------------------
Tool Calls:
Tool_Call_Name: multiply, Tool_Call_ID: 8efbdc8e-71fe-4a6d-b6bd-7f415a2c18a3, Arg: {'b': 50, 'a': 10}
Response MetaData:
model_name: gemini-3.1-flash-lite-preview, Finish Reason: STOP
Usage MetaData:
Input Tokens: 254, Output Tokens: 18, Total Tokens: 272
--------------------------------------------------
AI-content:  The result
Response MetaData:
model_name: None, Finish 